# AdPilot Pro – Model Comparison & Benchmarking Notebook

This notebook details the latency, size, and classification/regression metric comparison leaderboard for all AdPilot models.


## 1. Business Objectives



### Benchmarking Platform:
Evaluate and profile all sub-agent models (Strategy, Research, Content, Design, Analytics, Optimizer, CV) to confirm compliance with latency SLAs (< 100ms) and accuracy thresholds before promotion to production.



## 2. Load Models


In [ ]:
import joblib
import torch
import os
import sys
import pandas as pd
import numpy as np

# Load all exported weights
models_dir = "research/models"

# 1. Strategy Model
strategy_model = joblib.load("models/strategy/strategy_model.pkl")

# 2. Research Model
research_model = joblib.load(os.path.join(models_dir, "research", "research_model.pkl"))

# 3. Content Model
content_model = joblib.load(os.path.join(models_dir, "content", "content_model.pkl"))

# 4. Design Model
design_model = joblib.load(os.path.join(models_dir, "design", "creative_quality_score.pkl"))

# 5. Analytics Model
analytics_model = joblib.load(os.path.join(models_dir, "analytics", "analytics_model.pkl"))

# 6. Optimizer Model
import __main__
class PolicyNetwork(torch.nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(state_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, action_dim),
            torch.nn.Tanh()
        )
    def forward(self, x):
        return self.fc(x)
__main__.PolicyNetwork = PolicyNetwork

optimizer_model = joblib.load(os.path.join(models_dir, "optimizer", "optimizer_model.pkl"))

# 7. CV Model
cv_model = joblib.load(os.path.join(models_dir, "cv", "cv_model.pkl"))

print("All individual agent models loaded successfully.")


## 3. Model Size and Latency Profiling


In [ ]:
import time
import pandas as pd

# Load real features from the analytics dataset if available
feat_path = "research/datasets/processed/analytics_features.csv"
if os.path.exists(feat_path):
    df_feat = pd.read_csv(feat_path)
    columns = ["age", "balance", "duration", "campaign", "previous", "bal_dur_ratio", "campaign_efficiency"]
    analytics_row = df_feat[columns].iloc[0].values.reshape(1, -1)
else:
    analytics_row = np.array([[45, 10000.0, 300, 2, 1, 10000.0/301.0, 300*2]])

# Measure model size in KB and average latency in ms
models_list = [
    ("Strategy", strategy_model, np.array([[45, 10000.0, 300, 2, -1, 1, 1, 0, 1, 1, 1]])),
    ("Research", research_model, np.random.randn(1, 50)),
    ("Content", content_model, np.random.randn(1, 50)),
    ("Design", design_model, np.random.randn(1, 2)),
    ("Analytics", analytics_model, analytics_row),
    ("CV", cv_model, np.random.randn(1, 2))
]

leaderboard = []
for name, model, sample_input in models_list:
    # Measure Latency
    start_time = time.perf_counter()
    for _ in range(100):
        model.predict(sample_input)
    latency = (time.perf_counter() - start_time) * 10.0 # avg of 100 in ms
    
    # Measure size of pickle representation
    import pickle
    size_bytes = len(pickle.dumps(model))
    size_kb = size_bytes / 1024.0
    
    leaderboard.append({
        "Model Name": name,
        "Avg Latency (ms)": latency,
        "Size (KB)": size_kb
    })

leaderboard_df = pd.DataFrame(leaderboard).sort_values(by="Avg Latency (ms)")
print("AdPilot Agent Models Leaderboard:")
print(leaderboard_df)


## 4. Export Benchmark Report


In [ ]:
import os
import json
from datetime import datetime

out_dir = "research/reports/benchmarks"
os.makedirs(out_dir, exist_ok=True)

# Save markdown report
report_md = f"""# AdPilot Master Model Benchmark Report

* **Date**: {datetime.now().strftime('%Y-%m-%d')}
* **Strategy Size**: {leaderboard_df.loc[leaderboard_df['Model Name'] == 'Strategy', 'Size (KB)'].values[0]:.2f} KB
* **Content Latency**: {leaderboard_df.loc[leaderboard_df['Model Name'] == 'Content', 'Avg Latency (ms)'].values[0]:.4f} ms
"""

with open(os.path.join(out_dir, "benchmark_report.md"), "w") as f:
    f.write(report_md)

with open(os.path.join(out_dir, "benchmark_metrics.json"), "w") as f:
    json.dump(leaderboard, f, indent=2)

print("Benchmark reports successfully saved under research/reports/benchmarks/.")
